# Exercise 9, the Indian Ocean from gridded SST data

This exercise goes with **Lecture 9 &mdash; Evolution in time and space**. It is a
data-analysis workout with **xarray**: open a global sea-surface-temperature (SST)
dataset, select the Indian Ocean, and pull out the three things that matter most for the
South Asian monsoon &mdash; the **seasonal cycle**, the **warming trend**, and the
**Indian Ocean Dipole**. The finite-difference gradient in Exercise 3 is the same
operation you met in the lecture, now on real data.

The dataset is **NOAA ERSST v5** (monthly, 2° grid, 1970&ndash;2021), fetched through
`xarray.tutorial` &mdash; a few MB, downloaded once (Colab has internet).

Fill only the cells marked

```python
# ==== YOUR CODE ====
```


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

ds = xr.tutorial.open_dataset("ersstv5")
sst = ds["sst"]                       # (time, lat, lon), degC, monthly 1970-2021
print(sst)

def area_mean(da):
    """Area-weighted mean over lat/lon (cos-latitude weights)."""
    w = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(w).mean(("lat", "lon"))

# the Indian Ocean basin
indian_ocean = sst.sel(lat=slice(30, -30), lon=slice(30, 120))
print("\nIndian Ocean box:", dict(lat=(30, -30), lon=(30, 120)),
      "->", indian_ocean.shape)


## Exercise 1 &mdash; seasonal cycle and warming trend of the Indian Ocean

**Your task.**
1. Take the **area-weighted mean SST** of `indian_ocean` at every month (use the given
   `area_mean`). Call it `io_sst` &mdash; a 1-D time series.
2. Compute its **mean seasonal cycle** with `.groupby("time.month").mean()`.
3. Fit a straight line to `io_sst` versus time-in-years and report the **trend** in
   °C per decade.

**Hints.**
* `io_sst = area_mean(indian_ocean)`.
* Time in years: `yr = io_sst["time"].dt.year + (io_sst["time"].dt.month - 0.5) / 12`.
* `slope, intercept = np.polyfit(yr, io_sst, 1)`; trend per decade is `slope * 10`.


In [ ]:
# ==== YOUR CODE ====
io_sst   = None      # <-- area_mean(indian_ocean)
seasonal = None      # <-- io_sst.groupby("time.month").mean()

yr = None            # <-- io_sst["time"].dt.year + (io_sst["time"].dt.month - 0.5) / 12
slope = None         # <-- np.polyfit(yr, io_sst, 1)[0]
trend_per_decade = None   # <-- slope * 10
# ===================

print(f"Indian Ocean mean SST      : {float(io_sst.mean()):.2f} degC" if io_sst is not None else "")
print(f"warming trend              : {trend_per_decade} degC/decade")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if io_sst is not None:
    ax[0].plot(io_sst["time"], io_sst, lw=0.7, color="0.5")
    ax[0].plot(io_sst["time"], io_sst.rolling(time=12, center=True).mean(), color="tab:red", lw=1.5,
               label="12-month mean")
    if slope is not None:
        ax[0].plot(io_sst["time"], np.polyval(np.polyfit(yr, io_sst, 1), yr), "k--",
                   label=f"trend {trend_per_decade:.2f} degC/decade")
    ax[0].set_ylabel("SST [degC]"); ax[0].set_title("Indian Ocean mean SST")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
if seasonal is not None:
    ax[1].plot(seasonal["month"], seasonal, "o-")
    ax[1].set_xticks(range(1, 13)); ax[1].set_xlabel("month"); ax[1].set_ylabel("SST [degC]")
    ax[1].set_title("mean seasonal cycle"); ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. The seasonal cycle has two maxima and two minima, not one. Which months are the
   peaks, and why does the basin cool in July&ndash;August (think about what the summer
   monsoon winds do to the sea surface)?
2. What is the warming trend in °C/decade? Multiply by 5 &mdash; roughly how much has
   the Indian Ocean warmed since 1970? The Indian Ocean has warmed *faster* than the
   global ocean average; why does that matter for monsoon rainfall (Clausius&ndash;Clapeyron,
   Lecture 1)?
3. Detrend `io_sst` (subtract the fitted line). What is the standard deviation of the
   residual &mdash; the size of year-to-year SST swings &mdash; compared with the total
   warming so far?
```


## Exercise 2 &mdash; the Indian Ocean Dipole

The **Indian Ocean Dipole (IOD)** is the see-saw in SST between the western and eastern
tropical Indian Ocean. Its index, the **Dipole Mode Index (DMI)**, is the SST *anomaly*
in a western box minus the SST anomaly in a south-eastern box (Saji et al., 1999):

* **west box**: 50&ndash;70°E, 10°S&ndash;10°N
* **east box**: 90&ndash;110°E, 10°S&ndash;0°

A **positive** IOD (warm west, cool east) is associated with a stronger monsoon and
floods in East Africa; a **negative** IOD with the opposite.

**Your task.**
1. Select the two boxes (given). For each, compute the monthly **anomaly**
   (value minus that month's 1970&ndash;2021 climatology) and take its area-weighted mean.
2. `dmi = west_anom - east_anom`.
3. Plot the DMI, mark $\pm 1\sigma$, and print the calendar year with the strongest
   positive and strongest negative Sept&ndash;Nov (the IOD's peak season) value.

**Hints.**
* Anomaly: `anom = box.groupby("time.month") - box.groupby("time.month").mean()`.
* Then `area_mean(anom)`.
* SON season mean per year: `dmi.sel(time=dmi["time.month"].isin([9, 10, 11]))
  .groupby("time.year").mean()`.


In [ ]:
west_box = sst.sel(lat=slice(10, -10), lon=slice(50, 70))
east_box = sst.sel(lat=slice(0, -10),  lon=slice(90, 110))

# ==== YOUR CODE ====
west_anom = None     # <-- area_mean( west_box.groupby("time.month") - west_box.groupby("time.month").mean() )
east_anom = None     # <-- same for east_box
dmi = None           # <-- west_anom - east_anom

son = None           # <-- dmi.sel(time=dmi["time.month"].isin([9,10,11])).groupby("time.year").mean()
strongest_pos_year = None   # <-- int(son.idxmax())
strongest_neg_year = None   # <-- int(son.idxmin())
# ===================

if dmi is not None:
    print(f"DMI standard deviation : {float(dmi.std()):.2f} degC")
print(f"strongest positive IOD (SON): {strongest_pos_year}")
print(f"strongest negative IOD (SON): {strongest_neg_year}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(11, 4))
if dmi is not None:
    ax.plot(dmi["time"], dmi, lw=0.7, color="0.5")
    ax.plot(dmi["time"], dmi.rolling(time=5, center=True).mean(), color="tab:purple", lw=1.3)
    s = float(dmi.std())
    ax.axhline(s, color="tab:red", ls="--", lw=1, label="+1 sigma (positive IOD)")
    ax.axhline(-s, color="tab:blue", ls="--", lw=1, label="-1 sigma (negative IOD)")
    ax.fill_between(dmi["time"], s, dmi.where(dmi > s), color="tab:red", alpha=0.4)
    ax.fill_between(dmi["time"], -s, dmi.where(dmi < -s), color="tab:blue", alpha=0.4)
ax.set_ylabel("DMI [degC]"); ax.set_xlabel("year")
ax.set_title("Indian Ocean Dipole Mode Index")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. Which year has the strongest positive IOD? (It is famous &mdash; it coincided with
   severe East African floods and the Australian bushfire season.)
2. Is there a visible long-term trend in the DMI, or is it purely year-to-year noise?
   The frequency of strong positive events is projected to rise with warming &mdash; can
   you see the start of that in this record?
3. The DMI uses *anomalies*, not raw SST. Why would the raw west-minus-east SST
   difference be a poor index? (Think about the mean zonal gradient you will compute in
   Exercise 3.)
```


## Exercise 3 &mdash; the zonal SST gradient along the equator

The equatorial Indian Ocean is normally warm in the east and slightly cooler in the
west &mdash; a west-to-east *gradient* that the trade winds maintain by pushing warm
surface water eastward. A positive IOD flips the eastern end. Here you compute that
gradient with a finite difference, exactly as in Lecture 9.

**Your task.**
1. Take an equatorial band (5°S&ndash;5°N), average over latitude, to get SST as a
   function of longitude: `sst_x(lon)`.
2. Do this for the **long-term mean**, for **SON 2019** (strong positive IOD), and for
   **SON 1996** (near-neutral).
3. Compute the zonal gradient $\partial\,\text{SST}/\partial x$ of the long-term mean
   with `np.gradient`, converting the longitude axis to kilometres
   ($1° \approx 111\ \text{km}$ at the equator).

**Hints.**
* `band = sst.sel(lat=slice(5, -5), lon=slice(40, 110)).mean("lat")`.
* Long-term mean: `band.mean("time")`. Season mean:
  `band.sel(time=slice("2019-09", "2019-11")).mean("time")`.
* `x_km = band["lon"].values * 111.0`; `dSST_dx = np.gradient(mean_profile.values, x_km)`.


In [ ]:
band = sst.sel(lat=slice(5, -5), lon=slice(40, 110)).mean("lat")

# ==== YOUR CODE ====
mean_profile = None      # <-- band.mean("time")
prof_2019    = None      # <-- band.sel(time=slice("2019-09", "2019-11")).mean("time")
prof_1996    = None      # <-- band.sel(time=slice("1996-09", "1996-11")).mean("time")

x_km = None              # <-- band["lon"].values * 111.0
dSST_dx = None           # <-- np.gradient(mean_profile.values, x_km)   [degC per km]
# ===================

if dSST_dx is not None:
    print(f"mean zonal SST gradient (40-110E): {np.nanmean(dSST_dx)*1000:.3f} degC per 1000 km")
    print(f"east end (100E) SST: mean {float(mean_profile.sel(lon=100)):.2f}, "
          f"2019 {float(prof_2019.sel(lon=100)):.2f}, 1996 {float(prof_1996.sel(lon=100)):.2f}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
if mean_profile is not None:
    ax[0].plot(band["lon"], mean_profile, "k", lw=2, label="1970-2021 mean")
    ax[0].plot(band["lon"], prof_2019, color="tab:red", label="SON 2019 (positive IOD)")
    ax[0].plot(band["lon"], prof_1996, color="tab:blue", label="SON 1996 (neutral)")
    ax[0].set_xlabel("longitude [degE]"); ax[0].set_ylabel("SST [degC]")
    ax[0].set_title("equatorial SST vs longitude"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
if dSST_dx is not None:
    ax[1].plot(band["lon"], np.array(dSST_dx) * 1000)
    ax[1].axhline(0, color="0.6", lw=0.8)
    ax[1].set_xlabel("longitude [degE]"); ax[1].set_ylabel("dSST/dx [degC per 1000 km]")
    ax[1].set_title("zonal gradient of the mean (finite difference)"); ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. In the long-term mean, is the eastern equatorial Indian Ocean warmer or cooler than
   the western? What sign does that give $\partial\,\text{SST}/\partial x$?
2. How does the SON 2019 profile differ from the mean at the eastern end? That reversal
   *is* the positive IOD &mdash; relate it to the DMI value you found for 2019 in
   Exercise 2.
3. In Lecture 9, a gradient like this drives **advection**: warm water is carried down
   the temperature gradient by the current. Which way (east or west) would a wind-driven
   surface current carry heat here, and does that *reinforce* or *oppose* the mean
   gradient?
```


## Where this goes next

* The area-averaging, anomaly and `groupby` operations here are exactly the workflow of
  **Module 4** (processing reanalysis, satellite and CMIP6 data).
* The Indian Ocean Dipole and the SST gradient are two of the main climatic controls on
  the monsoon &mdash; the subject of **Module 5**.

```{note} Sources
Reworks the *analysing an ocean dataset* exercise of the
[*Climate of the Ocean*](https://github.com/florianboergel/climateoftheocean) course
for the Indian Ocean, for **CE524 Applied Hydroclimatology**. Data: NOAA Extended
Reconstructed SST v5 (Huang et al. 2017), no use constraints, via `xarray.tutorial`.
Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
